PART 1: COMBINED STRESS CALCULATIONS

Interfacial stress pipeline for PCB laminate moisture/thermal analysis

Implements the 5 key equations in Hygro-Thermo-Mechanical Modelling:
1. Fick's second law of diffusion        (governing PDE - solved in plane-sheet analytical mass uptake)
2. Plane-sheet analytical mass uptake    -> moisture_uptake_ratio()
3. Hygroscopic strain                    -> hygroscopic_strain()
4. Thermal mismatch strain                -> thermal_mismatch_strain()
5. Constrained comparative stress index  -> combined_stress_index()

Workflow of this notebook:
- **Part 1** defines the equations and `compute_stress()`.
- **Part 2** loads the material library (input parameters only) from `Formula_sheet_V2.xlsx`.
- **Part 3** runs the pipeline for the material grades you select.
- **Part 4** checks the notebook results against the values calculated in the Excel sheet.

In [1]:
# Import libraries
from pathlib import Path
import re

import numpy as np
import pandas as pd
import openpyxl

In [2]:
# Constants
MOLAR_MASS_WATER = 0.018015  # kg/mol, used to convert C_t from mol/m^3 to kg/m^3

In [3]:
# EQUATION 2 — Plane-sheet analytical mass uptake (solves Eq. 1, Fick's 2nd law)
def moisture_uptake_ratio(D, h, t, n_terms=50):
    """
    Mt/M_inf for 1D Fickian diffusion in a plane sheet (Crank's solution).
    D : diffusivity (m^2/s), h : thickness (m), t : time (s)
    """
    n = np.arange(n_terms)
    exponent = -((2 * n + 1) ** 2) * (np.pi ** 2) * D * t / (h ** 2)
    terms = (1.0 / (2 * n + 1) ** 2) * np.exp(exponent)
    return 1.0 - (8.0 / np.pi ** 2) * np.sum(terms)

In [4]:
# EQUATION 3 — Hygroscopic strain
def hygroscopic_strain(beta, W_kg_m3):
    """beta: CHE (m^3/kg), W: moisture concentration (kg/m^3)."""
    return beta * W_kg_m3

In [5]:
# EQUATION 4 — Thermal mismatch strain
def thermal_mismatch_strain(alpha_lam_ppm, alpha_cu_ppm, delta_T):
    """alpha values in 1e-6/K (ppm/K); delta_T in K."""
    alpha_lam = alpha_lam_ppm * 1e-6
    alpha_cu = alpha_cu_ppm * 1e-6
    return (alpha_lam - alpha_cu) * delta_T

In [6]:
# EQUATION 5 — Constrained comparative stress index
def combined_stress_index(E_eff, nu_eff, eps_hygro, eps_thermal):
    """E in MPa, nu dimensionless -> returns stress index in MPa."""
    return (E_eff / (1 - nu_eff)) * (eps_hygro + eps_thermal)

In [7]:
def compute_stress(D, Csat, h, t, beta, alpha_lam, alpha_cu, delta_T, E, nu, n_terms=50):
    """
    Runs all 5 equations for one manually-specified combination of inputs and
    returns the final stress index + intermediate values along the way.

    Inputs:
      D          : diffusivity (m^2/s)
      Csat       : saturation moisture concentration (mol/m^3)
      h          : laminate thickness (m)
      t          : conditioning time (s)
      beta       : hygroscopic swelling coefficient / CHE (m^3/kg)
      alpha_lam  : laminate CTE (ppm/K)
      alpha_cu   : copper CTE (ppm/K)
      delta_T    : temperature change for thermal mismatch (K)
      E          : effective modulus (MPa)
      nu         : Poisson's ratio (dimensionless)
      n_terms    : number of series terms for the moisture uptake sum (default 50)

    Returns a dict with:
      Mt_Minf           -- moisture uptake ratio (Eq. 2)
      C_t_mol_m3        -- moisture concentration at time t, mol/m^3
      W_kg_m3           -- moisture concentration at time t, kg/m^3
      eps_hygro         -- hygroscopic strain (Eq. 3)
      eps_thermal       -- thermal mismatch strain (Eq. 4)
      Stress_Index_MPa  -- combined stress index (Eq. 5)
    """
    mt_minf = moisture_uptake_ratio(D, h, t, n_terms=n_terms)
    C_t_mol_m3 = mt_minf * Csat
    W_kg_m3 = C_t_mol_m3 * MOLAR_MASS_WATER

    eps_h = hygroscopic_strain(beta, W_kg_m3)
    eps_t = thermal_mismatch_strain(alpha_lam, alpha_cu, delta_T)
    stress_index = combined_stress_index(E, nu, eps_h, eps_t)

    return {
        'Mt_Minf': mt_minf,
        'C_t_mol_m3': C_t_mol_m3,
        'W_kg_m3': W_kg_m3,
        'eps_hygro': eps_h,
        'eps_thermal': eps_t,
        'Stress_Index_MPa': stress_index,
    }

PART 2: MATERIAL LIBRARY

Material parameters are read from the Moisture and Thermal sheets of `Formula_sheet_V2.xlsx`. Only the **input** columns are loaded into the library; the calculations themselves are done in this notebook. Every material in the library comes from the formula sheet, so to add or change a material, edit the sheet and re-run from here.

In [8]:
# Settings
EXCEL_PATH = Path("Formula_sheet_V2.xlsx")   # change if the file is stored elsewhere
MOISTURE_SHEET = "Moisture"
THERMAL_SHEET = "Thermal"
HEADER_ROW = 4                              # row holding the column headers in both sheets

In [9]:
# Columns are found by the start of their header text in row 4, so the reader
# still works if columns in the sheet are moved or inserted.

# Descriptive columns (Moisture sheet)
INFO_COLS = {"supplier": "Supplier", "grade": "Grade", "class": "Class", "condition": "Condition"}

# Input parameters for compute_stress(): key = argument name
MOISTURE_INPUTS = {"D": "D (m2/s)", "Csat": "Csat", "h": "h (m)", "t": "t (s)"}
THERMAL_INPUTS = {"beta": "β", "alpha_lam": "αlam", "alpha_cu": "αCu",
                  "delta_T": "ΔT", "E": "Eeff", "nu": "νeff"}
INPUT_NAMES = list(MOISTURE_INPUTS) + list(THERMAL_INPUTS)

# Excel output columns - only used for the validation in Part 4
MOISTURE_OUTPUTS = {"Mt_Minf": "Mt/M∞ (fractional", "W_kg_m3": "W ="}
THERMAL_OUTPUTS = {"eps_hygro": "εh", "eps_thermal": "εT", "Stress_Index_MPa": "σ_index"}


def _find_columns(ws, wanted):
    """Return {key: column_index} by matching header text in HEADER_ROW."""
    headers = {c.column: str(c.value).strip() for c in ws[HEADER_ROW] if c.value is not None}
    found = {}
    for key, prefix in wanted.items():
        matches = [col for col, text in headers.items() if text.startswith(prefix)]
        if not matches:
            raise KeyError(f"Sheet '{ws.title}': no header starting with '{prefix}'")
        found[key] = matches[0]
    return found


_NUMERIC_EXPR = re.compile(r"^[0-9eE.+\-*/^() ]+$")

def _cell_value(ws_vals, ws_form, row, col):
    """
    Value of a cell. Uses the value Excel last calculated; if that is missing (file
    never recalculated), evaluates simple input formulas such as '=1.15*10^-13'
    or same-sheet references such as '=$C$20'.
    """
    v = ws_vals.cell(row, col).value
    if v is not None:
        return v
    f = ws_form.cell(row, col).value
    if not (isinstance(f, str) and f.startswith("=")):
        return f
    expr = f[1:].strip()
    ref = re.fullmatch(r"\$?([A-Z]{1,3})\$?(\d+)", expr)
    if ref:
        c = openpyxl.utils.column_index_from_string(ref.group(1))
        return _cell_value(ws_vals, ws_form, int(ref.group(2)), c)
    if _NUMERIC_EXPR.match(expr):
        return eval(expr.replace("^", "**"), {"__builtins__": {}})
    return None


def load_formula_sheet(path=EXCEL_PATH):
    """
    Returns two DataFrames with the same index:
      library        - descriptive columns + input parameters for compute_stress()
      excel_outputs  - the values the Excel sheet calculated (for validation only)
    Rows without a complete set of input parameters are left out of both.
    """
    wb_v = openpyxl.load_workbook(path, data_only=True)   # cached values
    wb_f = openpyxl.load_workbook(path, data_only=False)  # formulas (fallback only)
    mv, mf = wb_v[MOISTURE_SHEET], wb_f[MOISTURE_SHEET]
    tv, tf = wb_v[THERMAL_SHEET], wb_f[THERMAL_SHEET]

    m_cols = _find_columns(mf, {**INFO_COLS, **MOISTURE_INPUTS, **MOISTURE_OUTPUTS})
    t_cols = _find_columns(tf, {"grade": "Grade", "condition": "Condition",
                                **THERMAL_INPUTS, **THERMAL_OUTPUTS})

    lib_rows, out_rows, skipped = [], [], []
    for r in range(HEADER_ROW + 1, mf.max_row + 1):
        m = {k: _cell_value(mv, mf, r, c) for k, c in m_cols.items()}
        if m["grade"] is None:
            continue
        t = {k: _cell_value(tv, tf, r, c) for k, c in t_cols.items()}
        # The Thermal sheet uses the same row order; make sure it is the same material
        if (t["grade"], t["condition"]) != (m["grade"], m["condition"]):
            t = {k: None for k in t}

        params = {k: pd.to_numeric((m | t)[k], errors="coerce") for k in INPUT_NAMES}
        if any(pd.isna(v) for v in params.values()):
            skipped.append(f"row {r}: {m['grade']} {m['condition']}")
            continue
        lib_rows.append({"excel_row": r, **{k: m[k] for k in INFO_COLS}, **params})
        out_rows.append({k: pd.to_numeric((m | t)[k], errors="coerce")
                         for k in list(MOISTURE_OUTPUTS) + list(THERMAL_OUTPUTS)})

    if skipped:
        print("Skipped (incomplete inputs in sheet):", *skipped, sep="\n  ")
    return pd.DataFrame(lib_rows), pd.DataFrame(out_rows)


library, excel_outputs = load_formula_sheet()
print(f"Material library: {len(library)} grade/condition rows from {EXCEL_PATH.name}")

Material library: 52 grade/condition rows from Formula_sheet_V2.xlsx


In [10]:
# Material library - pick the index of the grade(s) you want to run
pd.set_option("display.max_rows", 200)
library[["excel_row", "supplier", "grade", "class", "condition"]]

,excel_row,supplier,grade,class,condition
0,5,Panasonic,CV8710U,EMC encapsulation,85°C/85%RH
1,6,Panasonic,CV8710U,EMC encapsulation,60°C/60%RH
2,7,Shengyi,SI10US/SI13U,Package substrate laminate/core,85°C/85%RH
3,8,Shengyi,SI10US/SI13U,Package substrate laminate/core,60°C/60%RH
4,9,Ajinomoto,ABF GX-T31,ABF build-up dielectric,85°C/85%RH
5,10,Ajinomoto,ABF GX-T31,ABF build-up dielectric,60°C/60%RH
6,11,Sumitomo Bakelite,EME-A700,EMC encapsulation,85°C/85%RH
7,12,Sumitomo Bakelite,EME-A700,EMC encapsulation,60°C/60%RH
8,13,Resonac,CEL-9200,EMC encapsulation,85°C/85%RH
9,14,Resonac,CEL-9200,EMC encapsulation,60°C/60%RH


PART 3: RUN THE STRESS CALCULATION PIPELINE

Select materials by index from the library above. Parameters in `OVERRIDES` replace the sheet values for every selected material, which lets you change test conditions (thickness, time, ΔT, etc.) from the notebook without editing the Excel file.

In [11]:
# Select material grades by index (from the library table)
SELECTED = [50, 51]           # e.g. [0, 1] = Panasonic CV8710U at 85°C/85%RH and 60°C/60%RH
# SELECTED = list(library.index)   # uncomment to run every grade

# Optional: override input parameters for all selected grades
OVERRIDES = {}              # e.g. {"t": 7 * 86400, "h": 0.0008}

In [12]:
def extract_parameters(idx, overrides=None, lib=library):
    """Return the compute_stress() inputs for the material at index idx."""
    params = {k: float(lib.loc[idx, k]) for k in INPUT_NAMES}
    unknown = set(overrides or {}) - set(INPUT_NAMES)
    if unknown:
        raise KeyError(f"Unknown override(s) {unknown}; valid names: {INPUT_NAMES}")
    params.update(overrides or {})
    return params


def run_selected(indices, overrides=None, lib=library):
    """Run compute_stress() for each selected index; return inputs + results."""
    rows = []
    for idx in indices:
        params = extract_parameters(idx, overrides, lib)
        rows.append({
            "index": idx,
            "material": f"{lib.loc[idx, 'supplier']} {lib.loc[idx, 'grade']}",
            "condition": lib.loc[idx, "condition"],
            **params,
            **compute_stress(**params),
        })
    return pd.DataFrame(rows).set_index("index")


results = run_selected(SELECTED, OVERRIDES)

# Input parameters used
results[["material", "condition", *INPUT_NAMES]]

,material,condition,D,Csat,h,t,beta,alpha_lam,alpha_cu,delta_T,E,nu
index,,,,,,,,,,,,
50,Nitto Denko GE-100,85°C/85%RH,1.900000e-13,215.0,0.0016,86400.0,0.00037,11.5,16.5,60.0,15000.0,0.26
51,Nitto Denko GE-100,60°C/60%RH,5.500000e-14,150.0,0.0016,86400.0,0.00037,11.5,16.5,35.0,15000.0,0.26


In [13]:
# Calculation results
results[["material", "condition", "Mt_Minf", "C_t_mol_m3", "W_kg_m3",
         "eps_hygro", "eps_thermal", "Stress_Index_MPa"]]

,material,condition,Mt_Minf,C_t_mol_m3,W_kg_m3,eps_hygro,eps_thermal,Stress_Index_MPa
index,,,,,,,,
50,Nitto Denko GE-100,85°C/85%RH,0.180717,38.854131,0.699957,0.000259,-0.000300,-0.831402
51,Nitto Denko GE-100,60°C/60%RH,0.097231,14.584601,0.262742,0.000097,-0.000175,-1.576735


PART 4: VALIDATION AGAINST THE EXCEL FORMULA SHEET

Compares the notebook results with the values the Excel sheet calculated for the same rows. The check is skipped if `OVERRIDES` is in use, since the Excel values were calculated with the sheet's own inputs.

In [14]:
REL_TOL = 1e-6   # relative tolerance

CHECK_LABELS = {"Mt_Minf": "Mt/M∞", "W_kg_m3": "W (kg/m³)", "eps_hygro": "εh",
                "eps_thermal": "εT", "Stress_Index_MPa": "σ_index (MPa)"}


def compare_with_excel(results, xl=excel_outputs, rel_tol=REL_TOL):
    rows = []
    for idx, r in results.iterrows():
        for col, label in CHECK_LABELS.items():
            nb_val, xl_val = r[col], xl.loc[idx, col]
            if pd.isna(xl_val):
                rel, status = np.nan, "no Excel value"
            else:
                rel = abs(nb_val - xl_val) / (abs(xl_val) if xl_val != 0 else 1.0)
                status = "PASS" if rel <= rel_tol else "CHECK"
            rows.append({"index": idx, "material": r["material"], "condition": r["condition"],
                         "quantity": label, "notebook": nb_val, "excel": xl_val,
                         "rel_diff": rel, "status": status})
    return pd.DataFrame(rows)


if OVERRIDES:
    print("Validation skipped: OVERRIDES is set, so results no longer correspond to the Excel inputs.")
else:
    comparison = compare_with_excel(results)
    n_check = (comparison["status"] == "CHECK").sum()
    print(f"{len(comparison)} values compared, {n_check} outside tolerance ({REL_TOL:g})")
    display(comparison)

10 values compared, 0 outside tolerance (1e-06)


,index,material,condition,quantity,notebook,excel,rel_diff,status
0,50,Nitto Denko GE-100,85°C/85%RH,Mt/M∞,0.180717,0.180717,2.610961e-15,PASS
1,50,Nitto Denko GE-100,85°C/85%RH,W (kg/m³),0.699957,0.699957,0.000000e+00,PASS
2,50,Nitto Denko GE-100,85°C/85%RH,εh,0.000259,0.000259,4.186365e-16,PASS
3,50,Nitto Denko GE-100,85°C/85%RH,εT,-0.000300,-0.000300,3.614007e-16,PASS
4,50,Nitto Denko GE-100,85°C/85%RH,σ_index (MPa),-0.831402,-0.831402,8.012172e-16,PASS
5,51,Nitto Denko GE-100,60°C/60%RH,Mt/M∞,0.097231,0.097231,1.570036e-15,PASS
6,51,Nitto Denko GE-100,60°C/60%RH,W (kg/m³),0.262742,0.262742,8.451064e-16,PASS
7,51,Nitto Denko GE-100,60°C/60%RH,εh,0.000097,0.000097,2.091130e-15,PASS
8,51,Nitto Denko GE-100,60°C/60%RH,εT,-0.000175,-0.000175,4.646581e-16,PASS
9,51,Nitto Denko GE-100,60°C/60%RH,σ_index (MPa),-1.576735,-1.576735,1.830732e-15,PASS


Notes on the validation:
- Excel uses the 4-term series when Fo ≥ 0.05 and the short-time solution when Fo < 0.05. The notebook uses a 50-term series throughout, which matches both well within tolerance at the Fo values in the sheet. Below Fo ≈ 1e-5, 50 terms are no longer enough.
- The Excel values are those last calculated when the workbook was saved. After editing inputs, save the workbook in Excel before re-running. A file saved by a script without recalculating will show "no Excel value".